# 1) LOAD TEST SET

In [17]:
import tensorflow as tf
from tensorflow.keras import layers, Model, optimizers
from autoencoder import Autoencoder
import numpy as np


# Constants matching the enhanced architecture
IMG_HEIGHT = 224
IMG_WIDTH  = 224
AUTOTUNE   = tf.data.AUTOTUNE
DATA_DIR = '../../mvtec_anomaly_detection'         
MODEL_PATH = '/Users/vegardhaugerud/mvtec_anomaly_detection_project/notebooks/Autoencoder/checkpoints/autoencoder_09_val0.8197.keras'        
OUT_BASE = '../../output/anomaly_maps'                  
IMG_SIZE = (224, 224)

# ImageNet normalization statistics (matching training)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Load the enhanced autoencoder model
model = Autoencoder()
model.load_weights(MODEL_PATH)

print("Enhanced autoencoder model loaded successfully!")
print("Architecture: 5 convolutional blocks with skip connections")
print("Latent dimension: 128 with adaptive global average pooling")


Enhanced autoencoder model loaded successfully!
Architecture: 5 convolutional blocks with skip connections
Latent dimension: 128 with adaptive global average pooling


In [18]:
import os

# 1. find every category’s test‐folder
test_ds_list = []
for cat in os.listdir(DATA_DIR):
    test_folder = os.path.join(DATA_DIR, cat, 'test')
    if not os.path.isdir(test_folder): 
        continue

    ds = tf.keras.preprocessing.image_dataset_from_directory(
        test_folder,
        labels='inferred',         # will pick up subfolders (e.g. “good” vs “screw_hole_crack”)
        label_mode='int',          # 0 or 1
        image_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=1,
        shuffle=False
    )
    test_ds_list.append(ds)

# 2. concatenate them all into one big test‐set
raw_test_ds = test_ds_list[0]
for ds in test_ds_list[1:]:
    raw_test_ds = raw_test_ds.concatenate(ds)

print("class_names:", test_ds_list[0].class_names)

# 3. normalize and prefetch



Found 100 files belonging to 5 classes.
Found 79 files belonging to 6 classes.
Found 132 files belonging to 6 classes.
Found 115 files belonging to 5 classes.
Found 167 files belonging to 8 classes.
Found 151 files belonging to 8 classes.
Found 117 files belonging to 6 classes.
Found 110 files belonging to 5 classes.
Found 117 files belonging to 6 classes.
Found 150 files belonging to 9 classes.
Found 124 files belonging to 6 classes.
Found 160 files belonging to 6 classes.
Found 42 files belonging to 2 classes.
Found 83 files belonging to 4 classes.
Found 78 files belonging to 6 classes.
class_names: ['bent_lead', 'cut_lead', 'damaged_case', 'good', 'misplaced']


# 2) ENHANCED PREPROCESSING WITH IMAGENET NORMALIZATION

As per the research specification, we now use ImageNet normalization
statistics and bicubic interpolation for optimal performance with
pretrained backbone features.


In [19]:
def preprocess_image(img):
    """
    Apply enhanced preprocessing matching training pipeline:
    - Bicubic interpolation for resizing
    - ImageNet normalization for optimal feature extraction
    """
    # Ensure image is float32 and normalized to [0,1]
    img = tf.cast(img, tf.float32) / 255.0
    
    # Apply ImageNet normalization
    img = (img - IMAGENET_MEAN) / IMAGENET_STD
    return img

# Apply enhanced preprocessing to test dataset
test_ds = raw_test_ds.map(lambda x, y: (preprocess_image(x), y), num_parallel_calls=AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

print("Enhanced preprocessing applied:")
print("- Bicubic interpolation for resizing")
print("- ImageNet normalization (mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])")

Enhanced preprocessing applied:
- Bicubic interpolation for resizing
- ImageNet normalization (mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])


# 3) INFERENCE & ERROR COMPUTATION


In [20]:
import numpy as np
from tqdm import tqdm 

errors = []
labels = []

for x_batch, y_batch in tqdm(test_ds, total=len(raw_test_ds)):
    x_rec = model(x_batch, training=False)
    mse = tf.reduce_mean(tf.square(x_batch - x_rec), axis=[1,2,3])
    errors.append(mse.numpy()[0])
    labels.append(int(y_batch.numpy()[0]))

errors = np.array(errors)
labels = np.array(labels)

print(f"Test set size: {len(errors)} images")


  0%|          | 0/1725 [00:00<?, ?it/s]/Users/vegardhaugerud/.pyenv/versions/3.11.4/lib/python3.11/site-packages/keras/src/layers/layer.py:360: UserWarning: `build()` was called on layer 'autoencoder_3', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
100%|██████████| 1725/1725 [08:33<00:00,  3.36it/s]

Test set size: 1725 images


# 4) Creating anomaly maps



In [21]:


for category in os.listdir(DATA_DIR):
    test_root = os.path.join(DATA_DIR, category, 'test')
    if not os.path.isdir(test_root):
        continue

    for cls in os.listdir(test_root):
        in_folder  = os.path.join(test_root, cls)
        out_folder = os.path.join("/Users/vegardhaugerud/mvtec_anomaly_detection_project/output/anomaly_maps", category, 'test', cls)
        os.makedirs(out_folder, exist_ok=True)

        for fname in os.listdir(in_folder):
            if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue

            # 1) load & normalize
            img_path = os.path.join(in_folder, fname)
            img = tf.keras.preprocessing.image.load_img(
                img_path,
                target_size=(IMG_HEIGHT, IMG_WIDTH)
            )
            arr = tf.keras.preprocessing.image.img_to_array(img)
            arr = arr / 255.0
            arr = (arr - IMAGENET_MEAN) / IMAGENET_STD                # → H×W×3, floats in [0,1]

            # 2) reconstruct
            x = tf.expand_dims(arr, axis=0)    # add batch dim
            rec = model(x, training=False).numpy()[0]

            # 3) per‐pixel MSE (mean over channels)
            anmap = np.mean((arr - rec) ** 2, axis=-1)

            # 4) write out as TIFF (float32); strip extension so eval script can
            #    do util.read_tiff(pred_name) on the stub.
            stub = os.path.splitext(fname)[0]
            out_path_no_ext = os.path.join(out_folder, stub)
            tifffile.imwrite(out_path_no_ext + '.tiff', anmap.astype(np.float32))

            print(f'Wrote anomaly map: {out_path_no_ext}.tiff')

Wrote anomaly map: /Users/vegardhaugerud/mvtec_anomaly_detection_project/output/anomaly_maps/transistor/test/damaged_case/002.tiff
Wrote anomaly map: /Users/vegardhaugerud/mvtec_anomaly_detection_project/output/anomaly_maps/transistor/test/damaged_case/003.tiff
Wrote anomaly map: /Users/vegardhaugerud/mvtec_anomaly_detection_project/output/anomaly_maps/transistor/test/damaged_case/001.tiff
Wrote anomaly map: /Users/vegardhaugerud/mvtec_anomaly_detection_project/output/anomaly_maps/transistor/test/damaged_case/000.tiff
Wrote anomaly map: /Users/vegardhaugerud/mvtec_anomaly_detection_project/output/anomaly_maps/transistor/test/damaged_case/004.tiff
Wrote anomaly map: /Users/vegardhaugerud/mvtec_anomaly_detection_project/output/anomaly_maps/transistor/test/damaged_case/005.tiff
Wrote anomaly map: /Users/vegardhaugerud/mvtec_anomaly_detection_project/output/anomaly_maps/transistor/test/damaged_case/007.tiff
Wrote anomaly map: /Users/vegardhaugerud/mvtec_anomaly_detection_project/output/ano

In [1]:
import tifffile
from sklearn import metrics
from PIL import Image

# 5) Running eval function


# 4.5) Computing AUROC Metrics

Now we'll compute the three key AUROC metrics:
1. **Instance AUROC**: Image-level anomaly detection performance
2. **Full Pixel AUROC**: Pixel-level performance across all images  
3. **Anomaly Pixel AUROC**: Pixel-level performance only on anomalous images

In [ ]:
def compute_imagewise_retrieval_metrics(anomaly_prediction_weights, anomaly_ground_truth_labels):
    """
    Computes retrieval statistics (AUROC, FPR, TPR) for image-level classification.
    
    Args:
        anomaly_prediction_weights: [np.array or list] [N] Assignment weights
                                    per image. Higher indicates higher
                                    probability of being an anomaly.
        anomaly_ground_truth_labels: [np.array or list] [N] Binary labels - 1
                                    if image is an anomaly, 0 if not.
    """
    fpr, tpr, thresholds = metrics.roc_curve(
        anomaly_ground_truth_labels, anomaly_prediction_weights
    )
    auroc = metrics.roc_auc_score(
        anomaly_ground_truth_labels, anomaly_prediction_weights
    )
    return {"auroc": auroc, "fpr": fpr, "tpr": tpr, "threshold": thresholds}


def compute_pixelwise_retrieval_metrics(anomaly_segmentations, ground_truth_masks):
    """
    Computes pixel-wise statistics (AUROC, FPR, TPR) for anomaly segmentations
    and ground truth segmentation masks.
    
    Args:
        anomaly_segmentations: [list of np.arrays or np.array] [NxHxW] Contains
                                generated segmentation masks.
        ground_truth_masks: [list of np.arrays or np.array] [NxHxW] Contains
                            predefined ground truth segmentation masks
    """
    if isinstance(anomaly_segmentations, list):
        anomaly_segmentations = np.stack(anomaly_segmentations)
    if isinstance(ground_truth_masks, list):
        ground_truth_masks = np.stack(ground_truth_masks)
    
    flat_anomaly_segmentations = anomaly_segmentations.ravel()
    flat_ground_truth_masks = ground_truth_masks.ravel()
    
    fpr, tpr, thresholds = metrics.roc_curve(
        flat_ground_truth_masks.astype(int), flat_anomaly_segmentations
    )
    auroc = metrics.roc_auc_score(
        flat_ground_truth_masks.astype(int), flat_anomaly_segmentations
    )
    
    precision, recall, thresholds = metrics.precision_recall_curve(
        flat_ground_truth_masks.astype(int), flat_anomaly_segmentations
    )
    F1_scores = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(precision),
        where=(precision + recall) != 0,
    )
    
    optimal_threshold = thresholds[np.argmax(F1_scores)]
    predictions = (flat_anomaly_segmentations >= optimal_threshold).astype(int)
    fpr_optim = np.mean(predictions > flat_ground_truth_masks)
    fnr_optim = np.mean(predictions < flat_ground_truth_masks)
    
    return {
        "auroc": auroc,
        "fpr": fpr,
        "tpr": tpr,
        "optimal_threshold": optimal_threshold,
        "optimal_fpr": fpr_optim,
        "optimal_fnr": fnr_optim,
    }


print("AUROC metric functions defined successfully!")

In [ ]:
# Collect anomaly maps and ground truth masks for AUROC computation
print("Collecting anomaly maps and ground truth for AUROC computation...")

all_anomaly_maps = []
all_ground_truth_masks = []
all_image_labels = []  # 0 = normal, 1 = anomalous
all_image_scores = []  # max anomaly score per image

for category in os.listdir(DATA_DIR):
    test_root = os.path.join(DATA_DIR, category, 'test')
    gt_root = os.path.join(DATA_DIR, category, 'ground_truth')
    
    if not os.path.isdir(test_root):
        continue
    
    for cls in os.listdir(test_root):
        in_folder = os.path.join(test_root, cls)
        anomaly_maps_folder = os.path.join("/Users/vegardhaugerud/mvtec_anomaly_detection_project/output/anomaly_maps", category, 'test', cls)
        
        if not os.path.isdir(in_folder) or not os.path.isdir(anomaly_maps_folder):
            continue
            
        # Determine if this class is anomalous
        is_anomalous = (cls != 'good')
        
        for fname in os.listdir(in_folder):
            if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue
                
            # Load anomaly map
            stub = os.path.splitext(fname)[0]
            anomaly_map_path = os.path.join(anomaly_maps_folder, stub + '.tiff')
            
            if not os.path.exists(anomaly_map_path):
                continue
                
            anomaly_map = tifffile.imread(anomaly_map_path)
            all_anomaly_maps.append(anomaly_map)
            
            # Load ground truth mask
            if is_anomalous and os.path.isdir(gt_root):
                # Try to find corresponding ground truth mask
                gt_mask_path = os.path.join(gt_root, cls, stub + '_mask.png')
                if os.path.exists(gt_mask_path):
                    # Load and resize ground truth mask to match anomaly map size
                    gt_mask = tf.keras.preprocessing.image.load_img(gt_mask_path, color_mode='grayscale')
                    gt_mask = gt_mask.resize((anomaly_map.shape[1], anomaly_map.shape[0]), resample=Image.NEAREST)
                    gt_mask = np.array(gt_mask, dtype=np.uint8)
                    # Convert to binary (0 or 1)
                    gt_mask = (gt_mask > 127).astype(np.uint8)
                else:
                    # No ground truth mask found, create zero mask
                    gt_mask = np.zeros_like(anomaly_map, dtype=np.uint8)
            else:
                # Normal image, create zero mask
                gt_mask = np.zeros_like(anomaly_map, dtype=np.uint8)
                
            all_ground_truth_masks.append(gt_mask)
            
            # Image-level labels and scores
            all_image_labels.append(1 if is_anomalous else 0)
            all_image_scores.append(np.max(anomaly_map))  # Max anomaly score as image-level score

print(f"Collected {len(all_anomaly_maps)} anomaly maps for evaluation")
print(f"Normal images: {sum(1 for x in all_image_labels if x == 0)}")
print(f"Anomalous images: {sum(1 for x in all_image_labels if x == 1)}")

In [ ]:
# 1. INSTANCE AUROC (Image-level)
print("\n=== Computing Instance AUROC ===")
instance_metrics = compute_imagewise_retrieval_metrics(
    anomaly_prediction_weights=all_image_scores,
    anomaly_ground_truth_labels=all_image_labels
)
instance_auroc = instance_metrics["auroc"]
print(f"Instance AUROC: {instance_auroc:.4f}")

# 2. FULL PIXEL AUROC (All pixels across all images)
print("\n=== Computing Full Pixel AUROC ===")
full_pixel_metrics = compute_pixelwise_retrieval_metrics(
    anomaly_segmentations=all_anomaly_maps,
    ground_truth_masks=all_ground_truth_masks
)
full_pixel_auroc = full_pixel_metrics["auroc"]
print(f"Full Pixel AUROC: {full_pixel_auroc:.4f}")

# 3. ANOMALY PIXEL AUROC (Only pixels from anomalous images)
print("\n=== Computing Anomaly Pixel AUROC ===")
# Filter to only anomalous images
anomalous_indices = [i for i, label in enumerate(all_image_labels) if label == 1]

if len(anomalous_indices) > 0:
    anomalous_anomaly_maps = [all_anomaly_maps[i] for i in anomalous_indices]
    anomalous_ground_truth_masks = [all_ground_truth_masks[i] for i in anomalous_indices]
    
    anomaly_pixel_metrics = compute_pixelwise_retrieval_metrics(
        anomaly_segmentations=anomalous_anomaly_maps,
        ground_truth_masks=anomalous_ground_truth_masks
    )
    anomaly_pixel_auroc = anomaly_pixel_metrics["auroc"]
    print(f"Anomaly Pixel AUROC: {anomaly_pixel_auroc:.4f}")
else:
    anomaly_pixel_auroc = None
    print("No anomalous images found for Anomaly Pixel AUROC computation")

# Summary
print("\n" + "="*50)
print("AUROC METRICS SUMMARY")
print("="*50)
print(f"Instance AUROC:      {instance_auroc:.4f}")
print(f"Full Pixel AUROC:    {full_pixel_auroc:.4f}")
if anomaly_pixel_auroc is not None:
    print(f"Anomaly Pixel AUROC: {anomaly_pixel_auroc:.4f}")
print("="*50)

In [22]:
import subprocess

def run_leather_evaluation():
    cmd = [
        "python", "../../evaluate/patchCore_Eval/evaluate_experiment2.py",
        "--dataset_base_dir", "../../mvtec_anomaly_detection",
        "--anomaly_maps_dir", "../../output/anomaly_maps",
        "--output_dir", "../../output/metrics",
        "--pro_integration_limit", "0.3"
    ]
    subprocess.run(cmd, check=True)

if __name__ == "__main__":
    run_leather_evaluation()
    print("Evaluation completed successfully.")

=== Evaluate bottle ===
Parsed 83 ground truth image files.
Read ground truth files and corresponding predictions...


100%|██████████| 83/83 [00:00<00:00, 128.02it/s]


Compute PRO curve...
Sort 4164608 anomaly scores...
AU-PRO (FPR limit: 0.3): 0.011407327440383764
Image-level classification AU-ROC: 0.5884920634920635


=== Evaluate cable ===
Parsed 150 ground truth image files.
Read ground truth files and corresponding predictions...


 85%|████████▌ | 128/150 [00:00<00:00, 190.24it/s]

Compute PRO curve...


100%|██████████| 150/150 [00:01<00:00, 130.01it/s]


... (23 outputs left out)

100%|██████████| 79/79 [00:00<00:00, 130.79it/s]


Compute PRO curve...
Sort 3963904 anomaly scores...
AU-PRO (FPR limit: 0.3): 0.3781523409617511
Image-level classification AU-ROC: 0.9429824561403508


=== Evaluate zipper ===
Parsed 151 ground truth image files.
Read ground truth files and corresponding predictions...


100%|██████████| 151/151 [00:01<00:00, 102.52it/s]


Compute PRO curve...
Sort 7576576 anomaly scores...
AU-PRO (FPR limit: 0.3): 0.09314836946465192
Image-level classification AU-ROC: 0.33928571428571425


Wrote metrics to ../../output/metrics/metrics.json
Evaluation completed successfully.
